# Belle 2018 model of $D^0 \to K^0_S \pi^+ \pi^-$

The amplitude model of the Belle 2018 analysis, in the BaBar 2008 parametrisation, built
as [SymPy](https://sympy.org) expressions with the AmpForm conventions
(`aman_examples.d02kspipi`). Every term agrees with the Belle C++ implementation to
$10^{-9}$ (`tests/test_cpp_reference.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Math
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.function.sympy import create_parametrized_function

from aman_examples.d02kspipi import COMPONENTS, formulate_model
from aman_examples.dynamics import LASS, BelleBreitWigner
from aman_examples.phasespace import DalitzPlotGenerator, is_inside

## Model

The amplitude is a coherent sum of quasi-two-body terms, each a coupling $C$ times a
lineshape and a Zemach spin factor $Z_J$. The kinematic variables are
$s_{ab} = m^2(K^0_S\pi^-)$ (`m2_kspim`) and $s_{bc} = m^2(K^0_S\pi^+)$ (`m2_kspip`).

In [ ]:
model = formulate_model()
Math(sp.latex(model.term("Kstar892minus")))

The lineshapes unfold to their definitions with `doit()` (here with `evaluate()`, one level only):

In [ ]:
bw = next(iter(model.components["Kstar892minus"].atoms(BelleBreitWigner)))
lass = next(iter(model.components["Kstarzero1430minus"].atoms(LASS)))
for expr in (bw, lass):
    display(Math(sp.latex(expr) + " = " + sp.latex(expr.evaluate())))

## Dalitz plot

The intensity $|A|^2$ is converted to a JAX function with
`tensorwaves.function.sympy.create_parametrized_function`. Its parameters (masses,
widths, couplings) can be changed without recompiling.

In [ ]:
intensity = create_parametrized_function(
    model.intensity.doit(), model.parameter_defaults, backend="jax"
)
masses = [model.parameter_defaults[model.masses[k]] for k in ("pi", "KS", "pi", "D")]
x = np.linspace(0.3, 3.1, 400)
X, Y = np.meshgrid(x, x, indexing="ij")
inside = is_inside(X, Y, *masses)
values = np.full(X.shape, np.nan)
values[inside] = intensity({"m2_kspim": X[inside], "m2_kspip": Y[inside]})

fig, ax = plt.subplots(figsize=(5.5, 4.5))
mesh = ax.pcolormesh(X, Y, np.log10(values), cmap="viridis")
fig.colorbar(mesh, ax=ax, label=r"$\log_{10} |A|^2$")
ax.set_xlabel(r"$m^2(K^0_S\pi^-)$ [GeV$^2$]")
ax.set_ylabel(r"$m^2(K^0_S\pi^+)$ [GeV$^2$]")
plt.show()

## Fit fractions

Fit fractions $\int |C_k A_k|^2 / \int |A|^2$, with the integrals estimated on
events uniform in the Dalitz plot (`aman_examples.phasespace.DalitzPlotGenerator`).
Interference terms are not included, so they do not sum to one.

In [ ]:
phsp = DalitzPlotGenerator(masses).generate(500_000, NumpyUniformRNG(seed=0))
total = np.mean(intensity(phsp))
for name in COMPONENTS:
    term = create_parametrized_function(
        sp.Abs(model.term(name)).doit() ** 2, model.parameter_defaults, backend="jax"
    )
    print(f"{name:20s} {np.mean(term(phsp)) / total:7.4f}")